In [ ]:
from PIL import Image

from ultralytics import YOLO

import sys
from pathlib import Path

# 프로젝트 루트 추가
ROOT = Path().resolve().parent   # fastapi-agent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 클래스만 import
from config.settings import Settings

# 인스턴스 생성 (env 자동 로드)
settings = Settings()

# yolo = YOLO("yolov8n.pt")
yolo = YOLO("yolov8n-oiv7.pt")


c:\xampp\htdocs\www\projects\fastapi-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [61]:
def detect_and_crop(image_path):
    img = Image.open(image_path).convert("RGB")

    r = yolo(image_path, conf=0.1)[0]

    # print("=======",r)
    # if len(r.boxes) == 0:
        # return None

    boxes = r.boxes.xyxy


    # print(boxes)
    # print("======================")

    confs = r.boxes.conf
    clss = r.boxes.cls

    idx = confs.argmax().item()

    x1, y1, x2, y2 = map(int, boxes[idx])


    # bounding box 크기
    w = x2 - x1
    h = y2 - y1

    img_x, img_y = img.size
    bbox_area = w * h
    img_area = img_x * img_y

    area_ratio = bbox_area / img_area


    min_area = 10

    if w < min_area or h < min_area:
        print(f"최소 크기 미만 : {img} | {min_area}")
        return None

    min_ratio = 0.01

    if area_ratio < min_ratio:
        print(f"최소 비율 미만 : {img} | {area_ratio}")
        return None


    crop = img.crop((x1, y1, x2, y2))

    # cw, ch = crop.size

    # if min(cw, ch) < 64:
    #     crop = img

    return {
        "crop": crop,
        "confidence": float(confs[idx]),
        "class_id": int(clss[idx]),
        "class_name": yolo.names[int(clss[idx])],
        "bbox": [x1, y1, x2, y2],
        "area_ratio": float(area_ratio)
    }

In [ ]:
img = ROOT / "storage" / "images" / "orange.jpg"

result = detect_and_crop(img)

print(result)

from sentence_transformers import SentenceTransformer
# embedding_model = SentenceTransformer('clip-ViT-B-32')
embedding_model = SentenceTransformer('clip-ViT-L-14')

embedded = embedding_model.encode(result['crop']).tolist()

from qdrant_client import AsyncQdrantClient
qdrant = AsyncQdrantClient()

rr = await qdrant.query_points(
    collection_name="fruits",
    query=embedded,
    limit=5,
)


from pprint import pprint

pprint(rr)

print(len(rr.points))
print("DONE")